### Giải Đa Thức (Cách ly & Newton)


In [ ]:
import numpy as np
from IPython.display import display, Math, Markdown

def format_poly(coeffs):
    n = len(coeffs) - 1
    terms = []
    for i, c in enumerate(coeffs):
        if abs(c) < 1e-10: continue
        deg = n - i
        sign = " + " if c > 0 else " - "
        c_abs = abs(c)
        c_str = f"{c_abs:.4f}" if abs(c_abs - 1.0) > 1e-10 or deg == 0 else ""
        if deg == 0: terms.append(f"{sign}{c_abs:.4f}")
        elif deg == 1: terms.append(f"{sign}{c_str}x")
        else: terms.append(f"{sign}{c_str}x^{deg}")
    res = "".join(terms).lstrip(" + ")
    if res.startswith("- "): res = "-" + res[2:]
    return res if res else "0"

def Polynomial_Solver_Full(coeffs_input, epsilon=1e-4):
    P = np.array(coeffs_input, dtype=float)
    display(Markdown("## ❖ TÌM TẤT CẢ NGHIỆM ĐA THỨC (CÁCH LY + NEWTON)"))
    display(Math(f"P(x) = {format_poly(P)} = 0"))
    
    # 1. TÌM KHOẢNG CÁCH LY (Sturm)
    display(Markdown("### 1. Phân tách khoảng cách ly nghiệm bằng định lý Sturm"))
    sturm_seq = [np.poly1d(P), np.polyder(np.poly1d(P))]
    while True:
        P_prev, P_curr = sturm_seq[-2], sturm_seq[-1]
        if P_curr.order == 0 and abs(P_curr.coeffs[0]) < 1e-10:
            sturm_seq.pop()
            break
        quot, rem = np.polydiv(P_prev, P_curr)
        P_next = np.poly1d(-rem.coeffs)
        if P_next.order == 0 and abs(P_next.coeffs[0]) < 1e-10: break
        sturm_seq.append(P_next)
        
    def count_sign_chànges(x_val):
        signs = [np.sign(p(x_val)) for p in sturm_seq if abs(p(x_val)) > 1e-10]
        return sum(1 for i in range(len(signs)-1) if signs[i]*signs[i+1] < 0)

    R = 1 + np.max(np.abs(P[1:])) / abs(P[0])
    step, start, end = 1.0, -np.ceil(R), np.ceil(R)
    
    intervals = []
    curr = start
    while curr <= end:
        a, b = curr, curr + step
        num_roots = count_sign_chànges(a) - count_sign_chànges(b)
        if num_roots == 1: intervals.append((a, b))
        elif num_roots > 1: # Chia nhỏ khoảng
            mid = (a + b) / 2.0
            if count_sign_chànges(a) - count_sign_chànges(mid) == 1: intervals.append((a, mid))
            if count_sign_chànges(mid) - count_sign_chànges(b) == 1: intervals.append((mid, b))
        curr += step
        
    for i, (a, b) in enumerate(intervals):
        display(Markdown(f"- Khoảng nghiệm thứ {i+1}: $x \in [{a:.4f}, {b:.4f}]$"))
        
    # 2. PHƯƠNG PHÁP NEWTON
    f = np.poly1d(P)
    df = np.polyder(f)
    d2f = np.polyder(df)
    
    for i, (a, b) in enumerate(intervals):
        display(Markdown(f"\n### 2.{i+1}. Giải nghiệm trong khoảng $[{a:.4f}, {b:.4f}]$ bằng Tiếp tuyến (Newton)"))
        x0 = a if f(a)*d2f(a) > 0 else b
        
        history = [x0]
        x_k = x0
        k = 0
        while True:
            k += 1
            x_new = x_k - f(x_k)/df(x_k)
            err = abs(x_new - x_k)
            history.append(x_new)
            if err < epsilon:
                break
            if k > 50:
                display(Markdown("Chưa hội tụ!"))
                break
            x_k = x_new
            
        display(Markdown(f"- **Tổng số lần lặp:** {k}"))
        display(Markdown(f"- **Sai số cuối cùng:** $\Delta = {err:.6e} \le \epsilon = {epsilon}$"))
        
        display(Markdown("- **Chi tiết các xấp xỉ:**"))
        if len(history) <= 5:
            for idx, val in enumerate(history):
                display(Math(f"x_{{{idx}}} = {val:.6f}"))
        else:
            display(Math(f"x_{{0}} = {history[0]:.6f} \\quad \\text{{(Xấp xỉ đầu)}}"))
            display(Math(f"x_{{1}} = {history[1]:.6f}"))
            display(Math(f"x_{{2}} = {history[2]:.6f}"))
            display(Markdown("$\\dots$"))
            display(Math(f"x_{{{k-2}}} = {history[-3]:.6f}"))
            display(Math(f"x_{{{k-1}}} = {history[-2]:.6f}"))
            display(Math(f"x_{{{k}}} = {history[-1]:.6f} \\quad \\text{{(Xấp xỉ cuối)}}"))
            
        # Kiểm tra lại nghiệm
        val_check = f(history[-1])
        display(Markdown("- **Kiểm tra nghiệm:**"))
        display(Math(f"P(x_{{{k}}}) = {val_check:.6e} \\approx 0 \\quad \\Rightarrow \\text{{Nghiệm hợp lệ!}}"))

# ========================================================
# NHẬP HỆ SỐ ĐA THỨC CỦA BẠN VÀO ĐÂY
# P(x) = a_n * x^n + a_{n-1} * x^{n-1} + ... + a_0
# Ví dụ: P(x) = 1*x^5 - 3*x^4 + 2*x^3 + 5*x^2 - x - 7

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
# He so da thuc theo thu tu tu bac cao nhat: [a_n, a_{n-1}, ..., a_1, a_0]
# Vi du: P(x) = x^5 - 2x^4 - 5x^3 + 4x^2 + 6x - 3
P_coeffs = [1.0, -2.0, -5.0, 4.0, 6.0, -3.0]
# ═══════════════════════════════════════════════════════════════════

Polynomial_Solver_Full(P_coeffs, epsilon=1e-6)
